# Process Model Matching on PMMC 2015: Data Preparation and Baselines

This notebook prepares the University Admission dataset from the Process Model Matching Contest 2015 and evaluates two baselines for activity matching:

1. **Feature-based classifier** – FLAN-T5 label descriptions, lexical and embedding similarities, and Gradient Boosting (the approach from my MS thesis).
2. **LLM, labels only** – Qwen2.5-7B-Instruct judges each candidate pair from the two activity labels.



## Setup

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentence-transformers networkx scikit-learn pandas tqdm

In [ ]:
import os, re, glob, zipfile, itertools, random, json, html, urllib.parse
import xml.etree.ElementTree as ET
from collections import defaultdict
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

## Data

University Admission set (9 BPMN models, 36 model pairs, gold-standard alignments) from the [Process Model Matching Contest 2015](https://ai.wu.ac.at/emisa2015/contest.php).

In [ ]:
DATA_URL = "https://ai.wu.ac.at/emisa2015/datasets/dataset1.zip"
DATA_DIR = "pmmc_admission"

if not os.path.exists("dataset1.zip"):
    !wget -q -O dataset1.zip "$DATA_URL"
with zipfile.ZipFile("dataset1.zip") as z:
    z.extractall(DATA_DIR)

bpmn_files = sorted(set(glob.glob(f"{DATA_DIR}/**/*.bpmn", recursive=True)
                        + glob.glob(f"{DATA_DIR}/**/*.bpmn20.xml", recursive=True)))
gold_files = sorted(glob.glob(f"{DATA_DIR}/**/*.rdf", recursive=True))
if not bpmn_files or not gold_files:
    print("Expected .bpmn and .rdf files were not found. Archive contents:")
    for f in sorted(glob.glob(f"{DATA_DIR}/**/*", recursive=True))[:60]:
        print("  ", f)
print(len(bpmn_files), "BPMN models:", [os.path.basename(f) for f in bpmn_files])
print(len(gold_files), "gold alignment files, e.g.", [os.path.basename(f) for f in gold_files[:3]])

## Process models as graphs

Each model becomes a directed graph: nodes are activities (with label and actor, taken from the lane or pool), and an edge links an activity to the next activity, passing through gateways and events.

In [ ]:
TASK_TAGS = {"task", "userTask", "manualTask", "serviceTask", "scriptTask", "businessRuleTask",
             "sendTask", "receiveTask", "subProcess", "callActivity"}

def local(tag):
    return tag.split("}")[-1]

def clean_label(s):
    s = html.unescape(s or "")
    s = re.sub(r"\s+", " ", s.replace("\\n", " ")).strip()
    return s

def parse_bpmn(path):
    """Activity graph of a BPMN model (node attributes: label, actor)."""
    root = ET.parse(path).getroot()
    all_nodes, tasks, flows = {}, {}, []
    process_name, node_process, node_lane = {}, {}, {}

    # pool names, used as actor when a task has no lane
    for el in root.iter():
        if local(el.tag) == "participant" and el.get("processRef"):
            process_name[el.get("processRef")] = clean_label(el.get("name"))

    for proc in root.iter():
        if local(proc.tag) != "process":
            continue
        pid = proc.get("id")
        for el in proc.iter():
            t, eid = local(el.tag), el.get("id")
            if t == "sequenceFlow":
                flows.append((el.get("sourceRef"), el.get("targetRef")))
            elif t == "lane":
                for ref in el:
                    if local(ref.tag) == "flowNodeRef" and ref.text:
                        node_lane[ref.text.strip()] = clean_label(el.get("name"))
            elif eid and t not in {"laneSet", "lane", "process", "documentation", "incoming", "outgoing"}:
                all_nodes[eid] = t
                node_process[eid] = pid
                if t in TASK_TAGS:
                    tasks[eid] = clean_label(el.get("name"))

    full = nx.DiGraph()
    full.add_nodes_from(all_nodes)
    full.add_edges_from((s, t) for s, t in flows if s and t)

    G = nx.DiGraph()
    for tid, label in tasks.items():
        actor = node_lane.get(tid) or process_name.get(node_process.get(tid), "") or "unknown"
        G.add_node(tid, label=label, actor=actor)
    # connect each activity to the next activity, skipping gateways and events
    for tid in tasks:
        seen, stack = {tid}, list(full.successors(tid)) if tid in full else []
        while stack:
            n = stack.pop()
            if n in seen:
                continue
            seen.add(n)
            if n in tasks:
                G.add_edge(tid, n)
            elif n in full:
                stack.extend(full.successors(n))
    return G

In [ ]:
def model_name(path):
    return re.sub(r"(\.bpmn20\.xml|\.bpmn|\.rdf)$", "", os.path.basename(path))

graphs = {model_name(f): parse_bpmn(f) for f in bpmn_files}
summary = pd.DataFrame([{"model": m, "activities": g.number_of_nodes(), "edges": g.number_of_edges(),
                         "actors": len({d['actor'] for _, d in g.nodes(data=True)})}
                        for m, g in graphs.items()])
display(summary)

ex = next(iter(graphs))
print(f"\nExample activities in {ex}:")
for n, d in list(graphs[ex].nodes(data=True))[:8]:
    print(f"  - {d['label']!r}  (actor: {d['actor']})")

## Gold-standard alignments

Alignments are stored in Alignment API (RDF) format. Entity URIs are resolved to activity ids, falling back to label matching where needed.

In [ ]:
RDF = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}"

def entity_keys(uri):
    """Candidate ids or labels encoded in an alignment URI."""
    u = urllib.parse.unquote(uri or "")
    keys = {u}
    for sep in ["#", "/"]:
        if sep in u:
            keys.add(u.rsplit(sep, 1)[-1])
    return {k.strip() for k in keys if k.strip()}

def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()

def build_lookup(G):
    by_id = {n: n for n in G.nodes}
    by_label = defaultdict(list)
    for n, d in G.nodes(data=True):
        by_label[norm(d["label"])].append(n)
    return by_id, by_label

LOOKUP = {m: build_lookup(g) for m, g in graphs.items()}

def resolve(uri, model):
    by_id, by_label = LOOKUP[model]
    for k in entity_keys(uri):
        if k in by_id:
            return by_id[k]
    for k in entity_keys(uri):
        hits = by_label.get(norm(k.replace("_", " ")), [])
        if len(hits) >= 1:
            return hits[0]
    return None

def pair_models_from_filename(path, models):
    """'Cologne-Frankfurt.rdf' -> ('Cologne', 'Frankfurt')"""
    stem = model_name(path)
    for a in models:
        for b in models:
            if a != b and stem in (f"{a}-{b}", f"{a}_{b}", f"{a}--{b}"):
                return a, b
    parts = re.split(r"-", stem)
    if len(parts) == 2 and all(p in models for p in parts):
        return parts[0], parts[1]
    return None

def parse_alignment(path):
    root = ET.parse(path).getroot()
    out = []
    for cell in root.iter():
        if local(cell.tag) != "Cell":
            continue
        e1 = e2 = None; rel = "="; measure = 1.0
        for ch in cell:
            t = local(ch.tag)
            res = ch.get(RDF + "resource") or ch.get("resource")
            if t == "entity1": e1 = res
            elif t == "entity2": e2 = res
            elif t == "relation" and ch.text: rel = ch.text.strip()
            elif t == "measure" and ch.text:
                try: measure = float(ch.text)
                except ValueError: pass
        if e1 and e2 and rel in ("=", "equivalence"):
            out.append((e1, e2, measure))
    return out

gold = {}          # (m1, m2) -> set of (id1, id2)
unresolved = 0
for f in gold_files:
    pm = pair_models_from_filename(f, graphs.keys())
    if pm is None:
        print("Skipping unrecognised model pair:", os.path.basename(f)); continue
    m1, m2 = pm
    s = set()
    for e1, e2, _ in parse_alignment(f):
        a, b = resolve(e1, m1), resolve(e2, m2)
        if a is None or b is None:          # some files list entities in reverse order
            a2, b2 = resolve(e2, m1), resolve(e1, m2)
            if a2 is not None and b2 is not None:
                a, b = a2, b2
        if a is None or b is None:
            unresolved += 1
        else:
            s.add((a, b))
    gold[(m1, m2)] = s

n_gold = sum(len(v) for v in gold.values())
print(f"Model pairs with gold: {len(gold)} | gold correspondences: {n_gold} | unresolved: {unresolved}")
assert n_gold > 0, "No gold pairs resolved – print a .rdf file and an entity URI to inspect the format."

## Pair dataset

All cross-model activity pairs for each model pair, labelled 1 if the pair appears in the gold standard.

In [ ]:
rows = []
for (m1, m2), gs in gold.items():
    g1, g2 = graphs[m1], graphs[m2]
    for a, da in g1.nodes(data=True):
        for b, db in g2.nodes(data=True):
            rows.append({"m1": m1, "m2": m2, "id1": a, "id2": b,
                         "label1": da["label"], "label2": db["label"],
                         "actor1": da["actor"], "actor2": db["actor"],
                         "y": int((a, b) in gs)})
pairs = pd.DataFrame(rows)
pairs["group"] = pairs["m1"] + "|" + pairs["m2"]
print(f"{len(pairs):,} activity pairs, {pairs.y.sum():,} gold matches ({pairs.y.mean():.2%})")
pairs[pairs.y == 1].sample(min(8, pairs.y.sum()), random_state=SEED)[["m1","label1","m2","label2"]]

## Baseline 1: feature-based classifier

FLAN-T5 expands each label and actor into a short description. Pairs are described by five features (string similarity, token overlap, label and description embedding similarity, same actor) and classified with Gradient Boosting. Evaluation uses 6-fold cross-validation grouped by model pair, so test pairs come from model pairs not seen in training.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

t5_name = "google/flan-t5-base"
t5_tok = AutoTokenizer.from_pretrained(t5_name)
t5 = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to(DEVICE).eval()

acts = {}
for m, g in graphs.items():
    for n, d in g.nodes(data=True):
        acts[(m, n)] = (d["label"], d["actor"])

def describe(batch):
    prompts = [f"Describe in one sentence what happens in this university admission process step: "
               f"'{lab}' (performed by: {act})." for lab, act in batch]
    enc = t5_tok(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = t5.generate(**enc, max_new_tokens=40)
    return t5_tok.batch_decode(out, skip_special_tokens=True)

keys = list(acts)
desc = {}
for i in tqdm(range(0, len(keys), 32), desc="T5 descriptions"):
    chunk = keys[i:i+32]
    for k, d in zip(chunk, describe([acts[k] for k in chunk])):
        desc[k] = d
for k in keys[:5]:
    print(acts[k][0], "->", desc[k])

In [ ]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

lab_emb = dict(zip(keys, emb_model.encode([acts[k][0] for k in keys], normalize_embeddings=True, batch_size=64)))
des_emb = dict(zip(keys, emb_model.encode([desc[k] for k in keys], normalize_embeddings=True, batch_size=64)))

def tokens(s): return set(norm(s).split())

def features(r):
    k1, k2 = (r.m1, r.id1), (r.m2, r.id2)
    t1, t2 = tokens(r.label1), tokens(r.label2)
    return {
        "seq_ratio": SequenceMatcher(None, norm(r.label1), norm(r.label2)).ratio(),
        "jaccard": len(t1 & t2) / max(1, len(t1 | t2)),
        "cos_label": float(lab_emb[k1] @ lab_emb[k2]),
        "cos_desc": float(des_emb[k1] @ des_emb[k2]),
        "same_actor": float(norm(r.actor1) == norm(r.actor2)),
    }

X = pd.DataFrame([features(r) for r in tqdm(pairs.itertuples(), total=len(pairs), desc="features")])
pairs = pd.concat([pairs.reset_index(drop=True), X], axis=1)
pairs[["seq_ratio","jaccard","cos_label","cos_desc","same_actor","y"]].groupby("y").mean()

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_recall_fscore_support

FEATS = ["seq_ratio", "jaccard", "cos_label", "cos_desc", "same_actor"]

def prf(y, p):
    P, R, F, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    return round(P, 3), round(R, 3), round(F, 3)

def best_threshold(y, s):
    grid = np.linspace(0.05, 0.95, 91)
    return max(grid, key=lambda t: prf(y, (s >= t).astype(int))[2])

results = []

# Single-feature thresholds (threshold tuned on training folds)
gkf = GroupKFold(n_splits=6)
for feat in ["seq_ratio", "cos_label", "cos_desc"]:
    pred = np.zeros(len(pairs), dtype=int)
    for tr, te in gkf.split(pairs, pairs.y, pairs.group):
        t = best_threshold(pairs.y.values[tr], pairs[feat].values[tr])
        pred[te] = (pairs[feat].values[te] >= t).astype(int)
    results.append(("Similarity threshold: " + feat, *prf(pairs.y, pred)))

# Feature-based classifier
pred = np.zeros(len(pairs), dtype=int); score = np.zeros(len(pairs))
for tr, te in gkf.split(pairs, pairs.y, pairs.group):
    w = np.where(pairs.y.values[tr] == 1, (pairs.y.values[tr] == 0).sum() / max(1, pairs.y.values[tr].sum()), 1.0)
    clf = GradientBoostingClassifier(random_state=SEED).fit(pairs[FEATS].values[tr], pairs.y.values[tr], sample_weight=w)
    s_tr = clf.predict_proba(pairs[FEATS].values[tr])[:, 1]
    t = best_threshold(pairs.y.values[tr], s_tr)
    score[te] = clf.predict_proba(pairs[FEATS].values[te])[:, 1]
    pred[te] = (score[te] >= t).astype(int)
pairs["gb_score"] = score
results.append(("Feature-based (T5 + Gradient Boosting)", *prf(pairs.y, pred)))

pd.DataFrame(results, columns=["Method", "Precision", "Recall", "F1"])

## Baseline 2: LLM, labels only

To keep inference tractable, candidates are retrieved first: for each activity, its top-3 neighbours in the other model by description similarity, in both directions. The LLM then answers Yes/No for each candidate, and P(Yes) is read from the next-token distribution. Non-candidate pairs count as negatives.

In [ ]:
TOP_K = 3
cand_idx = set()
for g, df in pairs.groupby("group"):
    for col_self, col_other in [("id1", "id2"), ("id2", "id1")]:
        for _, sub in df.groupby(col_self):
            cand_idx.update(sub.nlargest(TOP_K, "cos_desc").index)
pairs["candidate"] = pairs.index.isin(cand_idx)
cand = pairs[pairs.candidate].copy()
cand_recall = cand.y.sum() / pairs.y.sum()
print(f"{len(cand):,} candidate pairs, covering {cand_recall:.1%} of gold matches")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # Qwen2.5-3B-Instruct fits smaller GPUs
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(LLM_NAME)
tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto").eval()

YES = [tok.encode(w, add_special_tokens=False)[0] for w in ["Yes", " Yes", "yes"]]
NO  = [tok.encode(w, add_special_tokens=False)[0] for w in ["No", " No", "no"]]

SYSTEM = ("You are an expert in business process management. Two organizations describe their "
          "university admission processes. Decide if two activities mean the same thing.")

def prompt_label_only(r):
    q = (f"Activity A (organization 1): \"{r.label1}\"\n"
         f"Activity B (organization 2): \"{r.label2}\"\n\n"
         "Do A and B describe the same activity? Answer only Yes or No.")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def p_yes(prompts, bs=16):
    out = []
    for i in tqdm(range(0, len(prompts), bs), desc="LLM"):
        enc = tok(prompts[i:i+bs], return_tensors="pt", padding=True).to(llm.device)
        logits = llm(**enc).logits[:, -1, :].float()
        probs = torch.softmax(logits, -1)
        y, n = probs[:, YES].sum(-1), probs[:, NO].sum(-1)
        out.extend((y / (y + n + 1e-9)).cpu().tolist())
    return np.array(out)

print(prompt_label_only(next(cand.itertuples())))

In [ ]:
cand["llm_label_only"] = p_yes([prompt_label_only(r) for r in cand.itertuples()])
pairs["llm_label_only"] = 0.0
pairs.loc[cand.index, "llm_label_only"] = cand["llm_label_only"]

pred = (pairs["llm_label_only"] >= 0.5).astype(int)      # non-candidates count as negatives
results.append(("LLM, labels only (Qwen2.5-7B)", *prf(pairs.y, pred)))
results.append((f"Candidate recall, top-{TOP_K} (upper bound)", None, round(cand_recall, 3), None))
res_df = pd.DataFrame(results, columns=["Method", "Precision", "Recall", "F1"])
res_df

## Graph context

Neighbourhood used by the graph-context experiments: preceding activities, following activities and the actor.

In [ ]:
def graph_context(model, node, max_n=3):
    G = graphs[model]
    d = G.nodes[node]
    before = [G.nodes[p]["label"] for p in list(G.predecessors(node))[:max_n]]
    after  = [G.nodes[s]["label"] for s in list(G.successors(node))[:max_n]]
    return (f"Activity: \"{d['label']}\" | performed by: {d['actor']}\n"
            f"  comes after: {before or ['(start)']}\n"
            f"  followed by: {after or ['(end)']}")

r = pairs[pairs.y == 1].iloc[0]
print(graph_context(r.m1, r.id1)); print(); print(graph_context(r.m2, r.id2))

## Outputs

In [ ]:
pairs.to_csv(f"{OUT_DIR}/pairs.csv", index=False)
res_df.to_csv(f"{OUT_DIR}/results_baselines.csv", index=False)
json.dump({f"{a}|{b}": sorted(map(list, s)) for (a, b), s in gold.items()}, open(f"{OUT_DIR}/gold.json", "w"))
for m, g in graphs.items():
    nx.write_graphml(g, f"{OUT_DIR}/{m}.graphml")
!zip -qr outputs.zip {OUT_DIR}
print("Saved outputs/ and outputs.zip")
res_df